### Colab Activity 21.6: Hyperparameter Tuning with Keras

**Expected Time = 60 minutes**



This activity focuses on using hyperparameter tuning with the `keras` library.  There are two ways to perform a grid search with `keras`, and you will implement both. While `keras_tuner` was discussed in the lectures, here you will use the `Scikit-Learn` wrapper for keras to grid search the parameters using `GridSearchCV`.  You will implement this with the `KerasClassifier` to build some basic models on the wine dataset.  

#### Index

- [Problem 1](#-Problem-1)
- [Problem 2](#-Problem-2)
- [Problem 3](#-Problem-3)
- [Problem 4](#-Problem-4)

In [1]:
!pip install scikeras
import warnings

import pandas as pd

warnings.filterwarnings('ignore')
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.utils import to_categorical

2025-09-14 21:35:17.217234: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-14 21:35:17.228090: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757910917.242480  941423 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757910917.247278  941423 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-14 21:35:17.263901: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### The Data

Below, the wine dataset is loaded, split, and scaled.  

In [2]:
wine = load_wine(as_frame=True)

In [3]:
wine.frame.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
wine.frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

In [5]:
X = wine.data
y = to_categorical(wine.target)

In [6]:
X_scaled = StandardScaler().fit_transform(X)

[Back to top](#-Index)

### Problem 1

#### The Build Function



To use the `KerasClassifier` you first need to write a function that creates a `keras` model and takes in arguments for the parameters you wish to search. The pseudocode for this function is given below:

```python
def create_model(optimizer=..., neurons=..., activation=..., input_dim=...):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(neurons, activation=activation, input_shape=(input_dim,)))
    model.add(tf.keras.layers.Dense(1))  # Output layer for regression
    model.compile(optimizer=..., loss=....)
    return model
```

Your goal is to complete the definition of the `create_model` function using the arguments `optimizer = 'adam'` and `neurons=50` for `activation = 'relu'` and `input_dim = 13`. Inside the function, compile the model using the selected `optimizer` and `loss= 'mse'`.



In [19]:

tf.random.set_seed(42)


# Function to create a fully connected neural network model for SciKeras
def create_model(optimizer='adam', neurons=50, activation='relu', input_dim=13):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(neurons, activation=activation, input_shape=(input_dim,)))
    model.add(tf.keras.layers.Dense(1))  # Output layer for regression
    model.compile(
        optimizer=optimizer,
        loss='mse'
    )
    return model


### ANSWER CHECK
create_model

<function __main__.create_model(optimizer='adam', neurons=50, activation='relu', input_dim=13)>

[Back to top](#-Index)

### Problem 2

#### Creating the `KerasRegressos` model



Now, use the `create_model` function to instantiate `KerasRegressor` as `model` with  `verbose = 2`.


In [20]:

# Keras model with SciKeras wrapper
model = KerasRegressor(model=create_model)



[Back to top](#-Index)

### Problem 3

#### Performing the Grid Search



Now, to perform a grid search you just need to create a dictionary named `param_grid` with the hyperparameter `'model__neurons' : [10, 50, 100]`,     `'model__activation': ['relu', 'sigmoid']`,
`'model__optimizer': ['adam', 'sgd']`, `'batch_size': [1, 10]`, and
`'epochs': [10, 20]`.  

In [21]:

tf.random.set_seed(42)
# Hyperparameters to be optimized
param_grid = {
    'model__neurons': [10, 50, 100],
    'model__activation': ['relu', 'sigmoid'],
    'model__optimizer': ['adam', 'sgd'],
    'batch_size': [1, 10],
    'epochs': [10, 20]
}


[Back to top](#-Index)

### Problem 4

#### Fit and Evaluate the model



Use the `GridSearchCV` function with `estimator=model`, `param_grid=param_grid`, `scoring='neg_mean_squared_error'`, `cv=3`, and `verbose=2` to search your parameters and assign the results to `grid`. Next, use function `fit` on `grid` with the training data to fit your model.

In [22]:
# GridSearchCV for hyperparameter tuning
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3, verbose=2)
grid_result = grid.fit(X_scaled, y)

# Display the best hyperparameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


Fitting 3 folds for each of 48 candidates, totalling 144 fits
Epoch 1/10


I0000 00:00:1757911421.221469  947444 service.cc:148] XLA service 0x7cf9e0006510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757911421.221510  947444 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2025-09-14 21:43:41.231650: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1757911421.262598  947444 cuda_dnn.cc:529] Loaded cuDNN version 91001


 42/118 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0886

I0000 00:00:1757911421.430989  947444 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.9236
Epoch 2/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3807
Epoch 3/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2869
Epoch 4/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2602
Epoch 5/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2486
Epoch 6/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2423
Epoch 7/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2381
Epoch 8/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2351
Epoch 9/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2330
Epoch 10/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2316
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 927us/step
[CV] END batch_size=1, epochs=10, model__activation=relu, model__neurons=10, model__optimizer=adam; total time=   3.1s
Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 2.2180
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8908
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms

Finally, the results are written in a dataframe.

In [23]:

# Extract and display results from GridSearchCV
results = pd.DataFrame(grid_result.cv_results_)
print(results.head())

   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       2.578627      0.228440         0.155692        0.021034   
1       2.154278      0.020459         0.132197        0.004878   
2       2.279876      0.090433         0.137783        0.007885   
3       2.401011      0.331491         0.145088        0.016388   
4       2.287263      0.021716         0.146508        0.008457   

   param_batch_size  param_epochs param_model__activation  \
0                 1            10                    relu   
1                 1            10                    relu   
2                 1            10                    relu   
3                 1            10                    relu   
4                 1            10                    relu   

   param_model__neurons param_model__optimizer  \
0                    10                   adam   
1                    10                    sgd   
2                    50                   adam   
3                    50       

Because of the grading enviornment, a more exhaustive search over additional parameters is not an option.  To extend the work here should be straightforward enough, and this is a nice solution to grid searching the hyperparameters of a model.